In [1]:
import pandas as pd

# Load the two tables
merged_edges = pd.read_csv("raw_edges.csv")
blind_sum = pd.read_csv("blind_sum.csv")

# Keys identifying an edge
keys = ["Regulator", "Target", "Model", "Sign"]

# Columns that exist only in blind_sum
extra_columns = [c for c in blind_sum.columns if c not in merged_edges.columns]

print("Extra columns to transfer:")
print(extra_columns)

# Merge the extra columns onto merged_edges
result = merged_edges.merge(
    blind_sum[keys + extra_columns],
    on=keys,
    how="left",
)

# Find rows that exist only in blind_sum
new_rows = blind_sum.merge(
    merged_edges[keys],
    on=keys,
    how="left",
    indicator=True,
)

new_rows = new_rows[new_rows["_merge"] == "left_only"].drop(columns="_merge")

# Make sure the appended rows have all columns in the same order
for col in result.columns:
    if col not in new_rows.columns:
        new_rows[col] = pd.NA

new_rows = new_rows[result.columns]

# Append them
result = pd.concat([result, new_rows], ignore_index=True)

print(f"Original merged_edges : {len(merged_edges)} rows")
print(f"Rows added from blind_sum: {len(new_rows)}")
print(f"Final table: {len(result)} rows")

result.to_csv("merged_edges_with_blind_sum.csv", index=False)

display(result.head())

Extra columns to transfer:
['Classification', 'Confidence rank', 'Constraint', 'Notes']
Original merged_edges : 1587 rows
Rows added from blind_sum: 0
Final table: 1587 rows


,index,Regulator,Target,Sign,Model,Classification,Confidence rank,Constraint,Notes
0,1,AKT,AKT,positive,BL,unsupported,NaN,NaN,this inferred self-regulation is not supported...
1,2,EGFR,AKT,positive,BL,indirect,2,NaN,"causal effect mediated by PI3K, others"
2,3,HER2,AKT,positive,BL,indirect,2,NaN,"causal effect mediated by PI3K, others"
3,4,HER3,AKT,positive,BL,indirect,2,NaN,"causal effect mediated by PI3K, others"
4,5,PTEN,AKT,negative,BL,indirect,2,NaN,mediated by PIP3
